In [1]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')



# print(df.shape)   # (rows, columns) — e.g. (1000000, 79)
# print(df.head())    # First 5 rows
# print(df.columns)    # All column names

# # Check the last few columns — label is usually at the end
# print(df.columns.tolist())

# # Check columns that might be the label
# for col in df.columns:
#     unique_vals = df[col].nunique()
#     if unique_vals < 20:  # Labels usually have few unique values
#         print(f"{col}: {df[col].unique()}")

# Check if there's a readable version somewhere
# Try this first:
print(train_df.columns.tolist())
print(train_df.groupby('Label').size())
print(test_df.groupby('Label').size())

['ACK Flag Count', 'Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Average Packet Size', 'Avg Bwd Segment Size', 'Avg Fwd Segment Size', 'Bwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Header Length', 'Bwd IAT Max', 'Bwd IAT Mean', 'Bwd IAT Min', 'Bwd IAT Std', 'Bwd IAT Total', 'Bwd PSH Flags', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Bwd Packets/s', 'Bwd URG Flags', 'CWE Flag Count', 'Destination Port', 'Down/Up Ratio', 'ECE Flag Count', 'FIN Flag Count', 'Flow Bytes/s', 'Flow Duration', 'Flow IAT Max', 'Flow IAT Mean', 'Flow IAT Min', 'Flow IAT Std', 'Flow Packets/s', 'Fwd Avg Bulk Rate', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Header Length', 'Fwd Header Length.1', 'Fwd IAT Max', 'Fwd IAT Mean', 'Fwd IAT Min', 'Fwd IAT Std', 'Fwd IAT Total', 'Fwd PSH Flags', 'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 'Fwd Packets/s', 'Fwd

In [2]:


# Remove infinity values and fill nulls
# Separate features and labels for BOTH files
X_train = train_df.drop(columns=['Label', 'source'])
y_train = train_df['Label']

X_test = test_df.drop(columns=['Label', 'source'])
y_test = test_df['Label']

# Clean both
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Done!")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Done!
X_train: (2059411, 78)
X_test:  (514853, 78)
y_train: (2059411,)
y_test:  (514853,)


In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Scaling done!")

Scaling done!


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train_scaled, y_train)
print("Random Forest trained!")

Random Forest trained!


In [5]:
from sklearn.metrics import classification_report, accuracy_score

y_pred_rf = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.9979
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    429677
           1       0.99      1.00      0.99     18164
           2       1.00      0.71      0.83         7
           3       0.76      0.78      0.77       294
           4       0.45      0.38      0.42       130
           5       0.80      1.00      0.89         4
           6       1.00      1.00      1.00     25603
           7       1.00      1.00      1.00     34570
           8       1.00      1.00      1.00      2057
           9       0.99      0.99      0.99      1077
          10       0.99      0.99      0.99      1046
          11       1.00      1.00      1.00      1187
          12       1.00      1.00      1.00       644
          13       0.43      0.92      0.59       391
          14       1.00      1.00      1.00         2

    accuracy                           1.00    514853
   macro avg       0.89      0.92      0.90    514853
weighted 

In [6]:
import joblib

joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Saved!")

Saved!
